# Exercise Sheet 04: Knowledge Graphs and Information Extraction from Data

**Knowledge Representation Summer 2026**  
**Bielefeld University**  
**Adia Khalid, Benjamin Paaßen**  
**Exercise Sheet Publication Date: 2026-06-09**  
**Exercise Sheet Submission Deadline: Friday, 2026-06-19, noon (i.e. 23:59), via moodle**

**NOTE** The use of language models/AI tools is permitted under three conditions
1. transparency: you tell us that you used them
2. accountability: you take full responsibility for the submission, can explain and defend it
3. privacy: you do not transmit any private information to any external tool

## Setup — Install & Import Required Libraries

Run this cell **first** to install any missing packages.

In [1]:
# Install required packages (run once)
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ("spacy", "networkx", "matplotlib", "pandas"):
    install(pkg)

# Download spaCy English model
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])

print("\u2705 All packages are ready!")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
✅ All packages are ready!


---
## Part I: Knowledge Graph Reasoning

### Background

A **Knowledge Graph** is defined as $\mathcal{KG} = (\mathcal{E}, \mathcal{R}, \mathcal{F})$ where:
- $\mathcal{E}$ = set of **entities** (nodes)
- $\mathcal{R}$ = set of **relations** (edge types)
- $\mathcal{F}$ = set of **facts**, each a triple $\langle e_h,\, r,\, e_t \rangle$

**KG Reasoning** generates *new* facts from existing ones. The three core subtasks are:

| Subtask | Query form | What is missing? |
|---------|-----------|------------------|
| Tail completion | $\langle e_h,\, r,\, ? \rangle$ | the object |
| Head completion | $\langle ?,\, r,\, e_t \rangle$ | the subject |
| Relation prediction | $\langle e_h,\, ?,\, e_t \rangle$ | the relation |

**Logical rules** (Horn rules) are one way to fill these gaps.  
A Horn rule has the form: $A_1(X,Y) \wedge A_2(Y,Z) \wedge \ldots \rightarrow A_{head}(X,Z)$  
A rule is **closed** if every variable appears in at least two atoms.

### Task 4.1: KG Completion by hand

Below is a small fictional KG about scientists stored as a NumPy array.  
Each row is one triple: `[head_entity, relation, tail_entity]`.

Run the setup cell to see the triples, then answer the questions.

In [2]:
import numpy as np

# Small KG: each row = [head, relation, tail]
triples = np.array([
    ["Ada_Lovelace",   "BornInCity",   "London"],
    ["London",         "CityIn",        "England"],
    ["England",        "CountryIn",     "UK"],
    ["Ada_Lovelace",   "WorkedWith",    "Charles_Babbage"],
    ["Charles_Babbage","BornInCity",    "London"],
    ["Ada_Lovelace",   "KnownFor",      "First_Algorithm"],
    ["Charles_Babbage","KnownFor",      "Difference_Engine"],
    ["Alan_Turing",    "BornInCity",    "London"],
    ["Alan_Turing",    "WorkedAt",      "University_of_Manchester"],
    ["University_of_Manchester", "CityIn", "England"],
], dtype=str)

print(f"KG contains {len(triples)} triples\n")
print(f"{'Head':<28} {'Relation':<22} {'Tail'}")
print("-" * 65)
for h, r, t in triples:
    print(f"{h:<28} {r:<22} {t}")

KG contains 10 triples

Head                         Relation               Tail
-----------------------------------------------------------------
Ada_Lovelace                 BornInCity             London
London                       CityIn                 England
England                      CountryIn              UK
Ada_Lovelace                 WorkedWith             Charles_Babbage
Charles_Babbage              BornInCity             London
Ada_Lovelace                 KnownFor               First_Algorithm
Charles_Babbage              KnownFor               Difference_Engine
Alan_Turing                  BornInCity             London
Alan_Turing                  WorkedAt               University_of_Manchester
University_of_Manchester     CityIn                 England


**(a) Tail completion** — predict the missing tail:  
$\langle$ `Ada_Lovelace`, `Nationality`, `UK`  $\rangle$ 
Which triples did you chain together to reach your answer?

$\langle$ `Ada_Lovelace`, `BornInCity`, `London` $\rangle$  and  $\langle$  `London`, `CityIn`, `England` $\rangle$  and $\langle$  `England`, `CountryIn`, `UK` $\rangle$ 

Though for nationality also $\langle$ `Ada_Lovelace`, `Nationality`, `England`  $\rangle$ could be argued through $\langle$ `Ada_Lovelace`, `BornInCity`, `London` $\rangle$  and  $\langle$  `London`, `CityIn`, `England` $\rangle$ 

**(b) Relation prediction** — predict the missing relation:  
$\langle$ `Alan_Turing`, `Nationality`, `England` $\rangle$  
There is more than one valid path. Write out **one** path and the relation it implies.

From $\langle$ `Alan_Turing`, `BornInCity`, `London` $\rangle$  and $\langle$  `London`, `CityIn`, `England` $\rangle$ 

**(c) Head completion** — predict the missing head:  
$\langle$ `?`, `BornInCity`, `London` $\rangle$  
List **all** entities that satisfy this query.  

$\langle$ `Ada_Lovelace`, `BornInCity`, `London` $\rangle$ 
$\langle$ `Charles_Babbage`, `BornInCity`, `London` $\rangle$ 
$\langle$ `Alan_Turing`, `BornInCity`, `London` $\rangle$ 

What does having multiple answers tell us about the difficulty of head completion?

It is difficult and without further information, the right (or lookked-for) answer cannot be determined.

> **Hint:** For (a) and (b), follow the chain of relations step by step —  
> e.g. `BornInCity` → `CityIn` → `CountryIn` gives you a path from a person to a country.

In [47]:
# Helper: look up all triples where a specific column matches a value
# Use this to trace paths through the KG

def lookup(triples, col, value):
    """
    Return all rows where triples[:, col] == value.
    col: 0 = head, 1 = relation, 2 = tail
    """
    # TODO: use np boolean indexing to filter triples
    mask = triples[:, col] == value
    # e.g. lookup(triples, 0, "Ada_Lovelace") should return
    # all triples where Ada_Lovelace is the head
    return triples[mask]
    
# TODO (a): use lookup() to trace the chain and find Ada_Lovelace's nationality



searched_triple_a = lookup(lookup(triples, 0, "Ada_Lovelace"), 1, "BornInCity")

print(searched_triple_a[0][2])
print("-" * 65)
# TODO (b): use lookup() to find what connects Alan_Turing to England

everything_turing = lookup(triples, 0, "Alan_Turing")
print(everything_turing)
print("-" * 30)
everything_england = lookup(triples, 2, "England")
print(everything_england)
print("-" * 30)
res = []
# looking for everything that can connect turing to london in one step 
for tur in everything_turing:
    for eng in everything_england:
        if tur[2] == eng[0]:
            res.append(tur)
print(res)
print("-" * 30)
# Otherwise through manually looking up: 
searched_triple_b = lookup(lookup(triples, 0, "Alan_Turing"), 2, "London")
print(searched_triple_b[0][1])
searched_triple_b = lookup(lookup(triples, 0, "Alan_Turing"), 2, "University_of_Manchester")
print(searched_triple_b[0][1])
print("-" * 65)
# TODO (c): use lookup() to find all entities born in London

searched_triple_c = lookup(lookup(triples, 1, "BornInCity"), 2, "London")

for ent in searched_triple_c:
    print(ent[0])


London
-----------------------------------------------------------------
[['Alan_Turing' 'BornInCity' 'London']
 ['Alan_Turing' 'WorkedAt' 'University_of_Manchester']]
------------------------------
[['London' 'CityIn' 'England']
 ['University_of_Manchester' 'CityIn' 'England']]
------------------------------
[array(['Alan_Turing', 'BornInCity', 'London'], dtype='<U24'), array(['Alan_Turing', 'WorkedAt', 'University_of_Manchester'],
      dtype='<U24')]
------------------------------
BornInCity
WorkedAt
-----------------------------------------------------------------
Ada_Lovelace
Charles_Babbage
Alan_Turing


### Your answer

#### **(a) Tail completion — Ada_Lovelace Nationality ?**

Answer:   `London`
Chain used: `["Ada_Lovelace",   "BornInCity",   "London"]` -> `["London",         "CityIn",        "England"]` 

#### **(b) Relation prediction — Alan_Turing ? England**

Relation:  `BornInCity` & `WorkedAt`
Path: `['Alan_Turing' 'BornInCity' 'London']` -> `['London' 'CityIn' 'England']` & `['Alan_Turing' 'WorkedAt' 'University_of_Manchester']` -> `['University_of_Manchester' 'CityIn' 'England']`

#### **(c) Head completion — ? BornInCity London**

All matching entities:  `Ada_Lovelace`, `Charles_Babbage` & `Alan_Turing`
What this tells us about head completion: It is ambiguous. 

---
### Task 4.2: Horn Rules — read, apply, and write

**Horn rules** let us infer new triples automatically from existing ones.  
A rule is **closed** when every variable appears in at least two atoms — meaning no variable is left dangling.

Example of a closed Horn rule:  
$$\texttt{BornInCity}(X, Y) \wedge \texttt{CityIn}(Y, Z) \rightarrow \texttt{LivedIn}(X, Z)$$

Reading: *"If X was born in city Y, and Y is in country Z, then X lived in Z."*

**(a)** The rule below is given to you:

$$\texttt{BornInCity}(X, Y) \wedge \texttt{CityIn}(Y, Z) \wedge \texttt{CountryIn}(Z, W) \rightarrow \texttt{Nationality}(X, W)$$

Apply this rule to the KG from Task 4.1.  
List **all** new triples it generates. Write each as $\langle$ head, relation, tail $\rangle$.

**(b)** Is the rule in (a) a **closed** rule?  
Explain in one sentence — check whether every variable ($X$, $Y$, $Z$, $W$) appears in at least two atoms.

**(c)** Write your **own** Horn rule using relations from the KG that captures:  
*"If person X worked with person Y, and Y is known for achievement Z, then X is associated with Z."*  
Use the format: $\texttt{relation1}(X,Y) \wedge \texttt{relation2}(Y,Z) \rightarrow \texttt{relation3}(X,Z)$

> **Hint for (a):** For each entity that has a `BornInCity` triple, follow the chain  
> `BornInCity` → `CityIn` → `CountryIn` step by step using your `lookup()` function from Task 4.1.  
> **Hint for (b):** Count how many atoms each variable appears in. A variable appearing only once makes the rule non-closed.

In [63]:
def apply_nationality_rule(triples):
    """
    Apply the rule:
        BornInCity(X, Y) ^ CityIn(Y, Z) ^ CountryIn(Z, W) -> Nationality(X, W)

    Parameters
    ----------
    triples : np.ndarray, shape (n, 3)
        Each row is [head, relation, tail].

    Returns
    -------
    new_triples : np.ndarray, shape (m, 3)
        Inferred triples of the form [X, "Nationality", W].
    """
    new_triples = []

    # TODO 1: Find all triples where relation == "BornInCity".
    #         For each such triple [X, BornInCity, Y]:
    #           - look up triples where head == Y and relation == "CityIn" -> get Z
    #           - look up triples where head == Z and relation == "CountryIn" -> get W
    #           - if both steps succeed, append [X, "Nationality", W] to new_triples
    #         Hint: use your lookup() function from Task 4.1.
    born_in = lookup(triples, 1, "BornInCity")
    
    for trip in born_in:
        head_y_city =  lookup(lookup(triples, 0, trip[2]), 1, "CityIn")
        #print(head_y_city)
        if len(head_y_city) > 0:
            head_z_country  =  lookup(lookup(triples, 0, head_y_city[0][2]), 1, "CountryIn")
            #print(head_z_country)
            if len(head_z_country) > 0:
                new_triples.append([trip[0], "Nationality", head_z_country[0][2] ])
    # TODO 2: Convert new_triples to a numpy array and return it.
    #         If no new triples were found, return an empty array with shape (0, 3).
    return new_triples


inferred = apply_nationality_rule(triples)
print(f"Inferred {len(inferred)} new Nationality triples:\n")
for row in inferred:
    print(f"  <{row[0]}, {row[1]}, {row[2]}>")

Inferred 3 new Nationality triples:

  <Ada_Lovelace, Nationality, UK>
  <Charles_Babbage, Nationality, UK>
  <Alan_Turing, Nationality, UK>


### Your answer

#### **(a) New triples generated by the rule:**

`
 <Ada_Lovelace, Nationality, UK>
  <Charles_Babbage, Nationality, UK>
  <Alan_Turing, Nationality, UK>
`

#### **(b) Is the rule closed?**

Yes, every variable (X,Y,Z,W) appears in exactly two atoms. 

#### **(c) Your own Horn rule:**


In plain English this means: If a person X was born in city Y and city Y is in place Z und place Z ist in country W, then W ist the nationality of person X. 

---
## Part II: Named Entity Recognition (NER)

### Background

**NER** finds every segment in a text that mentions a named entity.  
Three main families of approaches:
- **Rule-based** — hand-crafted lexicons and patterns (high precision, domain-dependent)
- **Learning-based** — SVM, HMM, CRF (replace hand-crafted rules)
- **Neural-network-based** — BERT, ELMo (learn features automatically, state-of-the-art)

spaCy uses a **neural network pipeline** for NER and is one of the tools surveyed in Al-Moslmi et al. (2020).

### Task 4.3: NER with spaCy

Run the demo cell below, then answer in the answer cell:

**(a)** List every entity spaCy found and its type.  
**(b)** What do the labels **ORG**, **PERSON**, **GPE**, and **DATE** stand for?

 **Hint:** `spacy.explain(label)` prints the description for each label.  
  `en_core_web_sm` recognises 18 entity types. **GPE** = **G**eo-**P**olitical **E**ntity (countries, cities, states).

In [67]:
import spacy

# Load the English model (neural-network-based NER)
nlp = spacy.load("en_core_web_sm")

# Example sentence
text = "Apple CEO Tim Cook announced the new iPhone at the Steve Jobs Theater in Cupertino on June 5, 2023."

# Process the text through the spaCy pipeline
doc = nlp(text)

# Extract and display entities
print("Entities found:")
print("-" * 40)
for ent in doc.ents:
    print(f"  {ent.text:<30} \u2192 {ent.label_:<10} ({spacy.explain(ent.label_)})")

Entities found:
----------------------------------------
  Apple                          → ORG        (Companies, agencies, institutions, etc.)
  Tim Cook                       → PERSON     (People, including fictional)
  the Steve Jobs Theater         → FAC        (Buildings, airports, highways, bridges, etc.)
  Cupertino                      → GPE        (Countries, cities, states)
  June 5, 2023                   → DATE       (Absolute or relative dates or periods)


###  Your answer

#### **(a) Entities found:**

| Entity | Type | Description of type |
|--------|------|---------------------|
|    Apple    |   ORG   |       Companies, agencies, institutions, etc.              |
|   Tim Cook     |   PERSON   |  People, including fictional                 |
| the Steve Jobs Theater |FAC | Buildings, airports, highways, bridges, etc. | 
|    Cupertino    |    GPE  |        Countries, cities, states             |
| June 5, 2023   | DATE | Absolute or relative dates or periods | 


#### **(b) Label meanings:**

- **ORG** = Companies, agencies, institutions, etc.
- **PERSON** = People, including fictional
- **GPE** = Countries, cities, states
- **DATE** = Absolute or relative dates or periods

### Task 4.4: Try your own sentence

Write your own sentence about a **historical event** and run the NER code on it.  
Aim for a sentence that contains a **person**, a **place**, and a **year** — all three types should show up.

 **Hint — structure:** *"[Person] [did something] at [place] in [year]."*

In [68]:
# Task 4.4 -- write your own sentence here and answer the question below
my_text = "Angela Merkel was born in Hamburg in West Germany on the 17 July 1954"

# TODO: run my_text through the nlp pipeline and print each entity with its label
# (use the same loop pattern as the demo cell above)

doc = nlp(my_text)

print("Entities found:")
print("-" * 40)
for ent in doc.ents:
    print(f"  {ent.text:<30} \u2192 {ent.label_:<10} ({spacy.explain(ent.label_)})")

Entities found:
----------------------------------------
  Angela Merkel                  → PERSON     (People, including fictional)
  Hamburg                        → GPE        (Countries, cities, states)
  West Germany                   → GPE        (Countries, cities, states)
  July 1954                      → DATE       (Absolute or relative dates or periods)


### Your answer

#### **Your sentence:** Angela Merkel was born in Hamburg in West Germany on the 17 July 1954

#### **Entities detected** 
| Entity | Type | Type Description |
|---|---|---|
| Angela Merkel | PERSON | People, including fictional |
| Hamburg | GPE | Countries, cities, states |
| West Germany | GPE | Countries, cities, states |
| July 1954 | DATE | Absolute or relative dates or periods |



---
## Part III: Named Entity Disambiguation (NED)

> *"NED determines which named entity a mention refers to; e.g. 'Trump' can refer to a person, a corporation, or a building."* — Al-Moslmi et al. (2020)

The problem: the **same surface form** can map to **multiple real-world entities**. Two approaches:
- **Individual** — rank candidates by lexical similarity / co-occurrence with the mention.
- **Collective** — entities co-occurring in the same text are likely related (AGDISTIS, TagME).

Hossjer et al. (2022) connect this to **Bayesian inference**: we update our belief about which entity is meant by observing evidence (context words).

### Task 4.5: Bayesian disambiguation — teacher & student (Hossjer et al. 2022)

A teacher evaluates whether a child has learned addition. Three possible worlds:

| World | Interpretation | P(correct per question) |
|-------|----------------|-------------------------|
| $x_1$ | Student **guesses randomly** | $\pi_1 = 0.5$ |
| $x_2$ | Student **knows addition** | $\pi_2 = 0.8$ |
| $x_3$ | Student **cheats / gets help** | $\pi_3 = 0.8$ |

Proposition $S$: *"the child is expected to score well"* — true for $x_2$ and $x_3$.

#### **(a)** Why can the teacher **not distinguish** $x_2$ from $x_3$ based on test scores alone?  
####  **(b)** If the student scores **9 out of 10**, compute the **posterior probabilities** with a uniform prior $P_0(x_i)=1/3$ and a binomial likelihood.

> **Hint — Bayesian update:**
> $$P(x_i \mid d) = \frac{L(d \mid x_i)\,P_0(x_i)}{\sum_j L(d \mid x_j)\,P_0(x_j)}, \qquad L(d \mid x_i)=\binom{n}{k}\pi_i^{k}(1-\pi_i)^{n-k}$$
> With a uniform prior the $\binom{n}{k}$ cancels in the normalisation — just compute $\pi_i^{9}(1-\pi_i)^1$ for each world.  
> You may verify your numbers with the optional code cell below.

### Your answer

#### **(a)** Why $x_2$ and $x_3$ cannot be distinguished:

Because the teacher can only see that the student scored well and cannot determine between $x_2$ and $x_3$ based on that alone as `the child is expected to score well` is true for both cases. It would need more data to determine that. 

#### **(b) Calculation:** Type here or you can also add the image for the mathamatical calcualtion

- $L(d \mid x_1) = 0.009765625$
- $L(d \mid x_2) =0.26843545600000007$
- $L(d \mid x_3) =0.26843545600000007$
- Sum $=0.18221217900000003$
- $P(x_1 \mid d) = 0.01786493280086032$
- $P(x_2 \mid d) =0.4910675335995699$
- $P(x_3 \mid d) =0.4910675335995699$
- $P(S \mid d) = P(x_2 \mid d) + P(x_3 \mid d) =0.9821350671991398$

In [18]:
# verify your calculation numerically
from math import comb

n, k = 10, 9
pi    = [0.5, 0.8, 0.8]
prior = [1/3, 1/3, 1/3]

likelihoods = [comb(n, k) * p**k * (1 - p)**(n - k) for p in pi]
print("Likelihoods:", [f"{l:.5f}" for l in likelihoods])

# TODO:  compute posterior probabilities posterior[i] = (likelihood[i] * prior[i]) / sum(likelihood[j] * prior[j])
# ander ... P_xi_d = (likelihoods[i] * PO_xi) / (sum_j) mit sum_j sowas wie [likelihoods[i] * prior[i] for i in range(3)]

print("L(d|x_1) = " + str(likelihoods[0]))
print("L(d|x_2) = " + str(likelihoods[1]))
print("L(d|x_3) = " + str(likelihoods[2]))
sum_j =  sum(likelihoods[i] * prior[i] for i in range(3))
print("sum = " + str(sum_j)  )

p_x1_d =  (likelihoods[0] * prior[0]) / (sum_j)
print("P(x_1|d) = " +  str(p_x1_d) )
p_x2_d = (likelihoods[1] * prior[1]) / (sum_j)
print("P(x_2|d) = " + str( p_x2_d ))
p_x3_d = (likelihoods[2] * prior[2]) / (sum_j)
print("P(x_3|d) = " + str( p_x3_d ))

print("P(S|d) = " + str( p_x2_d + p_x3_d))


Likelihoods: ['0.00977', '0.26844', '0.26844']
L(d|x_1) = 0.009765625
L(d|x_2) = 0.26843545600000007
L(d|x_3) = 0.26843545600000007
sum = 0.18221217900000003
P(x_1|d) = 0.017864932800860323
P(x_2|d) = 0.4910675335995699
P(x_3|d) = 0.4910675335995699
P(S|d) = 0.9821350671991398


### Task 4.6: Disambiguating "Washington"

The word *Washington* can refer to many real-world entities. NED uses **context** to pick the right one.

Match each sentence to the correct entity, and say **which clue word(s)** decided it.

| Sentence | Entity (A–D) | Clue word(s) |
|----------|--------------|--------------|
| *"Washington crossed the Delaware."* | | |
| *"The flight to Washington was delayed."* | | |
| *"Washington defeated Oregon 31–20."* | | |
| *"She applied to Washington for her PhD."* | | |

**Candidates:** A) George Washington (person) · B) Washington, D.C. (city) · C) University of Washington (organisation) · D) Washington Commanders (sports team)

> **Hint (collective NED):** *"what is mentioned in the same text tends to be about the same topic."* Look for the other content words in each sentence.

### Your answer

Fill in the **Entity** and **Clue word(s)** columns in the table above (or rewrite them here):

- *"Washington crossed the Delaware."* →
- *"The flight to Washington was delayed."* →
- *"Washington defeated Oregon 31–20."* →
- *"She applied to Washington for her PhD."* →

---
## Part IV: Named Entity Linking (NEL)

> *"NEL annotates each mention with the IRI of its corresponding entity in a knowledge base."* — Al-Moslmi et al. (2020)

Key distinction:
- **NER** → *type* level: `Apple` = ORG
- **NEL** → *instance* level: `Apple` = `https://www.wikidata.org/entity/Q312` (Apple Inc.)

Wikidata uses **Q-IDs** (e.g. `Q937` for Albert Einstein).

### Task 4.7: Linking entities to Wikidata

#### **(a)** Look up each entity at **https://www.wikidata.org** and record its Q-ID.

| Entity | Wikidata ID (Q…) |
|--------|-------------------|
| Albert Einstein | |
| Berlin | |
| COVID-19 | |
| Marie Curie | |
| The Beatles | |

####  **(b)** Write the **RDF triple** for *"Albert Einstein is a person"* using the Q-ID you found, the predicate `rdf:type`, and a suitable object.

**Hint:** Entity IRI pattern: `https://www.wikidata.org/entity/Q<number>` · `rdf:type` = `http://www.w3.org/1999/02/22-rdf-syntax-ns#type`. A triple is `<subject> <predicate> <object> .`

### Your answer 

#### **(a)** Q-IDs — fill in the table above.

#### **(b)** Your triple:

```
<YOUR TRIPLE HERE>
```

---
## Part V: LLMs for Information Extraction

### Background (Zhu et al. 2024)

Zhu et al. evaluated LLMs on KG construction (NER, relation extraction) and reasoning (link prediction, QA). Key finding:

> *"LLMs, represented by GPT-4, are more suited as inference assistants rather than few-shot information extractors."*

Performance on KG construction (F1):

| Model | DuIE2.0 | Re-TACRED |
|-------|---------|-----------|
| ChatGPT (0-shot) | 10.26 | 15.2 |
| GPT-4 (0-shot) | 31.03 | 15.5 |
| GPT-4 (1-shot) | 41.91 | 22.5 |
| Fine-tuned SOTA | 69.4 | 69.4 |

### Task 4.8: Prompt engineering for relation extraction

For the sentence *"Marie Curie discovered radium in Paris in 1898."*:

####  **(a)** Write a prompt that asks an LLM to extract triples.  
####  **(b)** What triples do you **expect** it to output?  
####  **(c)** According to you what is the **main limitation** of LLMs as information extractors vs. fine-tuned models?

 **Hints:** Be explicit about output format (e.g. JSON list of `[subject, predicate, object]`); ask for triples only (no preamble); one example (1-shot) roughly doubles F1. One sentence can yield several triples (who / what / where / when). For (c), look at the F1 gap in the table above.

###  Your answer 

####  **(a) Your prompt:**

```
[write your prompt here]
```

####  **(b) Expected triples:**

####  **(c) Main limitation :**

### Task 4.9: Virtual Knowledge (VINE) + Learning vs. Knowledge Acquisition

The VINE experiment tests whether LLMs **memorise** pre-training facts or can **generalise** to brand-new knowledge injected via instructions.

> *"Learning increases true belief; **knowledge acquisition** additionally requires the true belief to be **justified** — formed for the right reason."* — Hossjer et al. (2022)

### **(a)** VINE results showed GPT-4 far above ChatGPT. What does that tell us about **memorising vs. generalising**?  
### **(b)** Back to the teacher in Task 4.5: after seeing 9/10, the teacher has *learned* that $S$ is more likely. What **additional evidence** would turn that learning into **knowledge** that the student genuinely knows addition?

**Hint for (b):** Knowledge needs justification — evidence that pushes the *cheating* world $x_3$ toward probability 0 (e.g. observing the student under supervision).

### Your answer 

####  **(a) Memorise vs. generalise:**

####  **(b) Evidence that turns learning into knowledge:**

---
## References

- **Al-Moslmi et al. (2020).** Named entity extraction for knowledge graphs: a literature overview. *IEEE Access* 8, 32862–32881.
- **Hossjer, Diaz-Pachon & Rao (2022).** A formal framework for knowledge acquisition: going beyond machine learning. *Entropy* 24(10), 1469.
- **Zeng, Cheng & Si (2023).** Logical rule-based knowledge graph reasoning: a comprehensive survey. *Mathematics* 11(21), 4486.
- **Zhu et al. (2024).** LLMs for knowledge graph construction and reasoning. *World Wide Web.* doi:10.1007/s11280-024-01297-w.
- **spaCy:** https://spacy.io/ · **Wikidata:** https://www.wikidata.org

---
